# Module 5 — Similarity: Entity Resolution and Chunk-Similarity Linking

**The gap, from Module 4:** extraction stores every mention as its own `RecognisedEntity`,
keyed by `(string, doc_id)`. The same physical entity now exists many times over — "3M Company",
"3M", and "MMM" all appear as separate nodes from the same filing, "3M Company" appears
*again* in every other 3M filing under the same string, and a curated `Company {id: "3M"}` node
from Module 1 exists independently of all of them. Module 3's Wikidata enrichment adds a third
source of the same kind of duplication for people and subsidiaries. Separately, the same
near-identical *text* — boilerplate legal language, a paragraph repeated across a company's own
filing years — appears many times over too, wasting retrieval budget on redundant chunks.

**What we build in this module, in two parts:**
- **Entity resolution** (`similarity/resolver.py`) — per entity type, narrow candidates by name
  similarity, enrich survivors with graph context, ask an LLM to confirm which are truly the
  same real-world entity, and write the result as one `EntityGroup` node per confirmed cluster
  with `SAME_AS` edges from every member
- **Chunk-similarity linking** (`similarity/linker.py`) — find near-duplicate `Chunk`s via the
  `chunk_embedding` vector index, including across documents and companies, and store them as
  `SIMILAR_TO {score}` edges
- A redundancy-aware re-rank (`similarity/rerank.py`) that uses that precomputed similarity
  graph to drop near-duplicate chunks before they reach retrieval grading
- An exploratory look (`similarity/clusters.py`) at what Louvain community detection adds on
  top of the similarity graph

**New components introduced:**
- `similarity.candidates` — fuzzy name-matching to narrow entity candidates (`find_fuzzy_candidates`)
- `similarity.context` — enriches a candidate entity with its graph neighborhood (`fetch_entity_context`)
- `similarity.resolver` — LLM-judged entity resolution and `EntityGroup`/`SAME_AS` writing (`resolve_entity_type`, `resolve_all_entities`, `judge_candidates`)
- `similarity.validators` — structured-output schema for LLM resolution judgments (`ResolutionResult`)
- `similarity.linker` — chunk-to-chunk similarity search and `SIMILAR_TO` writing (`link_similar_chunks`)
- `similarity.rerank` — similarity-aware deduplication of a retrieved chunk list (`deduplicate_by_similarity`)
- `similarity.clusters` — exploratory Louvain topic clustering over an ephemeral GDS projection (`detect_topic_clusters`)
- `ingestion.schema.apply_similarity_schema` — the `EntityGroup.id` uniqueness constraint
- `agent.graph.build_agent(dedup=True)` — wires `deduplicate_by_similarity` into the retrieval loop as a new node

## 1. The duplicate problem, in the graph as it stands

Module 4 ran its extraction demo over four chunks of 3M's filings (see `module_04_extraction.ipynb`,
section 6) plus Module 3's Wikidata enrichment ran for both 3M and Apple. Look at what "3M" alone
resolves to today: several `RecognisedEntity` mentions across two different filings, plus a
separate curated `Company` node — none of them connected to each other.


In [1]:
from financial_advisor.services.neo4j_service import neo4j_service

rows = neo4j_service.run_query(
    """
    MATCH (e:RecognisedEntity {type: "Company"})
    WHERE e.string CONTAINS "3M"
    RETURN "RecognisedEntity" AS label, e.string AS name, e.doc_id AS context
    UNION
    MATCH (c:Company)
    WHERE c.id CONTAINS "3M"
    RETURN "Company" AS label, c.id AS name, "curated (Module 1/3)" AS context
    """
)
for row in rows:
    print(f"{row['label']:16s} | {row['name']:45s} | {row['context']}")
print(f"\n{len(rows)} node(s), zero of them connected to each other yet")


RecognisedEntity | 3M Company                                    | 3M/3M_2025_10K.pdf
RecognisedEntity | 3M                                            | 3M/3M_2025_10K.pdf
RecognisedEntity | 3M Company                                    | 3M/3M_2024_10K.pdf
RecognisedEntity | 3M Financial Management Company               | 3M/3M_2024_10K.pdf
RecognisedEntity | 3M Innovative Properties Company              | 3M/3M_2024_10K.pdf
RecognisedEntity | 3M Interamerica LLC                           | 3M/3M_2024_10K.pdf
RecognisedEntity | 3M Chemical Operations LLC                    | 3M/3M_2024_10K.pdf
RecognisedEntity | 3M Fall Protection Company                    | 3M/3M_2024_10K.pdf
RecognisedEntity | 3M Foreign Holding LLC                        | 3M/3M_2024_10K.pdf
RecognisedEntity | 3M do Brasil Ltda.                            | 3M/3M_2024_10K.pdf
RecognisedEntity | 3M Belgium BV                                 | 3M/3M_2024_10K.pdf
RecognisedEntity | 3M Canada Company - Compagnie 3M Ca

## 2. A naive fix: substring matching

The most obvious approach: pick a target, find every other node whose name contains it (or vice
versa), and merge them all. No graph context, no LLM — just string containment. This is
notebook-only, deliberately worse than what ships in `src/` (same convention as Module 4's naive
prompt demo).


In [2]:
# Ad hoc, throwaway heuristic — NOT how similarity.resolver works. Exists to be visibly wrong.
naive_matches = neo4j_service.run_query(
    """
    MATCH (e:RecognisedEntity {type: "Company"})
    WHERE e.string CONTAINS "3M"
    RETURN e.string AS name
    ORDER BY name
    """
)
print(f"Naive substring match on '3M' pulls in {len(naive_matches)} entities, e.g.:")
for row in naive_matches[:8]:
    print(f"  - {row['name']}")
print("  ...")


Naive substring match on '3M' pulls in 34 entities, e.g.:
  - 3M
  - 3M Belgium BV
  - 3M Canada Company - Compagnie 3M Canada
  - 3M Chemical Operations LLC
  - 3M China Limited
  - 3M Company
  - 3M Company
  - 3M Deutschland GmbH
  ...


**What went wrong.** Every one of 3M's ~40 consolidated subsidiaries also contains "3M" in its
name — "3M Financial Management Company", "3M Chemical Operations LLC", "3M do Brasil Ltda.", and
so on. A naive fix would merge the parent company with all of its subsidiaries into one entity,
which is exactly wrong: `SUBSIDIARY_OF` is a real, meaningful relationship in this graph
(Module 3's Wikidata structure, Module 4's extracted subsidiaries table) that a careless merge
would destroy.


## 3. Adding a real candidate filter: fuzzy matching

`similarity/candidates.py::find_fuzzy_candidates` replaces substring containment with RapidFuzz's
`WRatio` scorer — better at matching real name variants (abbreviations, corporate suffixes, word
order) — but it's still just comparing strings. Run it against the same target and the result
looks almost identical to the naive attempt: subsidiaries share too much of the parent's name to
be filtered out by string similarity alone.


In [3]:
from financial_advisor.extraction.validators import EntityType
from financial_advisor.similarity.candidates import fetch_candidate_pool, find_fuzzy_candidates

company_pool = fetch_candidate_pool(EntityType.COMPANY)
print(f"Unresolved Company-type pool: {len(company_pool)} node(s)")

target = next(n for n in company_pool if n.name == "3M Company")
rest = [n for n in company_pool if n is not target]
fuzzy_matches = find_fuzzy_candidates(target, rest, threshold=78, limit=8)

print(f"\nFuzzy candidates for '{target.name}':")
for m in fuzzy_matches:
    print(f"  - [{m.label:16s}] {m.name}")


Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `SAME_AS` does not exist in database `pack-course-bck`. Verify that the spelling is correct.', position=<SummaryInputPosition line=3, column=25, offset=74>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 74, 'line': 3, 'column': 25}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        MATCH (e:RecognisedEntity {type: $type})\n        WHERE NOT (e)-[:SAME_AS]->(:EntityGroup)\n        RETURN e.string AS string, e.doc_id AS doc_id\n        '
Received notification from DBMS server: <GqlStatusObject gql_status='01N50', status_description='warn: label does not exist.

Unresolved Company-type pool: 91 node(s)

Fuzzy candidates for '3M Company':
  - [RecognisedEntity] 3M Company
  - [RecognisedEntity] 3M
  - [Company         ] 3M
  - [RecognisedEntity] 3M Financial Management Company
  - [RecognisedEntity] 3M Innovative Properties Company
  - [RecognisedEntity] 3M Interamerica LLC
  - [RecognisedEntity] 3M Chemical Operations LLC
  - [RecognisedEntity] 3M Fall Protection Company


Still no way to tell, from names alone, that "3M Financial Management Company" is a subsidiary
and "3M" (the short form used elsewhere in the same filing) is not. That distinction only exists
in the graph — as a `RELATED_TO {type: "SUBSIDIARY_OF"}` edge Module 4 already extracted.


## 4. What graph context adds

`similarity/context.py::fetch_entity_context` enriches a candidate into the JSON the LLM actually
sees: the node's own properties, plus its 1-hop neighborhood in any direction (any relationship
type, skipping bulky `Chunk`/`Document` nodes), plus — for extracted mentions — which company's
filing it was mentioned in. For "3M Company" itself, the neighborhood includes the regulatory and
location context extracted alongside it; for a subsidiary like "3M Financial Management Company",
it includes the `SUBSIDIARY_OF` edge back to the parent — the one piece of evidence a name-only
comparison can never see.


In [4]:
import json

from financial_advisor.similarity.context import fetch_entity_context

target_context = fetch_entity_context(target)
print("Target — '3M Company':")
print(json.dumps(target_context, indent=2)[:900], "...\n")

subsidiary = next(m for m in fuzzy_matches if m.name == "3M Financial Management Company")
subsidiary_context = fetch_entity_context(subsidiary)
print("Candidate — '3M Financial Management Company':")
print(json.dumps(subsidiary_context, indent=2))


Target — '3M Company':
{
  "name": "3M Company",
  "type": "Company",
  "source": "RecognisedEntity",
  "own_properties": {
    "string": "3M Company",
    "mentions": [
      "3M Company",
      "the Company",
      "3M"
    ],
    "description": "Company described as a diversified technology company with a global presence.",
    "type": "Company",
    "doc_id": "3M/3M_2025_10K.pdf"
  },
  "neighbors": [
    {
      "relationship": "RELATED_TO",
      "direction": "out",
      "node_label": "RecognisedEntity",
      "name": "subsidiaries",
      "node_type": "Company"
    }
  ],
  "filed_in_filings_of": [
    "3M"
  ]
} ...

Candidate — '3M Financial Management Company':
{
  "name": "3M Financial Management Company",
  "type": "Company",
  "source": "RecognisedEntity",
  "own_properties": {
    "string": "3M Financial Management Company",
    "mentions": [
      "3M Financial Management Company"
    ],
    "description": "Listed as a consolidated subsidiary of the Registrant.",
    "t

## 5. LLM judgment: confirm or reject, with a reason

`similarity/resolver.py::judge_candidates` sends the target's context plus every fuzzy
candidate's context to the LLM in one structured-output call
(`similarity.validators.ResolutionResult`) and gets back a `same_entity` verdict + one-sentence
reason per candidate — grounded in the graph evidence just built, not name similarity.


In [5]:
from financial_advisor.similarity.resolver import judge_candidates

candidate_contexts = [fetch_entity_context(m) for m in fuzzy_matches]
result = judge_candidates(EntityType.COMPANY, target_context, candidate_contexts)

for j in result.judgments:
    verdict = "SAME" if j.same_entity else "different"
    print(f"[{verdict:9s}] {fuzzy_matches[j.index].name:40s} — {j.reason}")


[SAME     ] 3M Company                               — The candidate is explicitly named 3M Company and described as the filer/registrant, matching the target exactly.
[SAME     ] 3M                                       — The candidate is 3M and its mentions include "The Company," which is the same entity as 3M Company in the target.
[SAME     ] 3M                                       — The candidate has id "3M" and the same company attributes (ticker MMM, founded 1902, NYSE), indicating it is 3M Company.
[different] 3M Financial Management Company          — This is a consolidated subsidiary of the Registrant, so it is a different company from 3M Company.
[different] 3M Innovative Properties Company         — This is a consolidated subsidiary of the Registrant, not the parent 3M Company.
[different] 3M Interamerica LLC                      — This entity is listed as a consolidated subsidiary of 3M Company, so it is not the same company.
[different] 3M Chemical Operations LLC        

**An illustrative outcome, one run:** the LLM confirmed the three genuine duplicates — "3M
Company" (another filing), "3M" (the short form used in the same filing), and the curated
`Company {id: "3M"}` node from Module 1/3 — and rejected every subsidiary, each time citing the
`SUBSIDIARY_OF` edge as the reason it's a distinct entity. That's the payoff of the graph-context
step: the same name-similarity signal that made subsidiaries *candidates* in the first place can
be correctly overruled once the model can see how they actually relate to the parent — though as
with any LLM judgment, the exact verdicts and reasons worded here can vary between runs.

## 6. Storage: `EntityGroup` and `SAME_AS`

Confirmed clusters get written as one `EntityGroup {id, type, canonical_name}` node with a
`SAME_AS` edge from every member (target + confirmed candidates). `canonical_name` prefers a
curated node's name over an extracted mention, since it's already clean (Wikidata-sourced or the
company's own `id`). No group is written when nothing is confirmed — a target with zero confirmed
matches just stays ungrouped and gets reconsidered on the next run.

`apply_similarity_schema()` adds the one new constraint this needs: uniqueness on
`EntityGroup.id`.

In [6]:
from financial_advisor.ingestion.schema import apply_similarity_schema

apply_similarity_schema()


  [schema] OK  entity_group_id
[schema] 1/1 statements applied — all good


## 7. Running the pipeline for real

`resolve_entity_type` ties steps 3-6 together for one entity type; `resolve_all_entities` runs it
across every type in `RESOLUTION_ORDER` — companies and people first (richest graph context,
curated counterparts to reconcile against), then the extraction-only types. Idempotent: a node
already under a `SAME_AS` edge is excluded from the pool up front, so rerunning this after a
future extraction batch only processes what's new.


In [7]:
from financial_advisor.similarity.resolver import resolve_all_entities

summary = resolve_all_entities()
print(f"\nGroups created per type: {summary}")
print(f"Total: {sum(summary.values())}")

#It can take a while - we can use this time for a small break

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `SAME_AS` does not exist in database `pack-course-bck`. Verify that the spelling is correct.', position=<SummaryInputPosition line=3, column=25, offset=74>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 74, 'line': 3, 'column': 25}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        MATCH (e:RecognisedEntity {type: $type})\n        WHERE NOT (e)-[:SAME_AS]->(:EntityGroup)\n        RETURN e.string AS string, e.doc_id AS doc_id\n        '
Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type doe

[resolve:Company] 91 unresolved node(s)
[resolve:Company] '3M Company' (RecognisedEntity) — 8 fuzzy candidate(s): ['3M Company', '3M', '3M', '3M Financial Management Company', '3M Innovative Properties Company', '3M Interamerica LLC', '3M Chemical Operations LLC', '3M Fall Protection Company']
[resolve:Company] group d52e7bd0 = 3M Company + ['3M Company', '3M', '3M']
[resolve:Company] 'The Company's' (RecognisedEntity) — 2 fuzzy candidate(s): ['The Company', 'the Company']
[resolve:Company] group fb9959fb = The Company's + ['The Company', 'the Company']
[resolve:Company] 'Solventum' (RecognisedEntity) — 1 fuzzy candidate(s): ['Solventum Corporation']
[resolve:Company] group 49278603 = Solventum + ['Solventum Corporation']
[resolve:Company] 'the U.S. Environmental Protection Agency' (RecognisedEntity) — 1 fuzzy candidate(s): ['3M Fall Protection Company']
[resolve:Company] 'the U.S. Environmental Protection Agency' — LLM confirmed none, no group created
[resolve:Company] '3M Financial M

### Check the database
Let's check our database using the browser at [http://localhost:7474/browser/](http://localhost:7474/browser/)

## 8. Inspecting the result

Every `EntityGroup` and its members, across all types — this is the reconciled view Module 6/7
can query instead of guessing which raw `RecognisedEntity`/`Company`/`Person` nodes refer to the
same thing.


In [ ]:
rows = neo4j_service.run_query(
    """
    MATCH (g:EntityGroup)<-[:SAME_AS]-(member)
    RETURN g.type AS type, g.canonical_name AS canonical_name,
           collect(coalesce(member.string, member.name, member.id)) AS members
    ORDER BY type, canonical_name
    """
)
for row in rows:
    print(f"[{row['type']:16s}] {row['canonical_name']:30s} <- {row['members']}")
print(f"\n{len(rows)} EntityGroup(s) total")


## 9. Where entity resolution leaves the graph

Running this over the corpus produces a set of `EntityGroup`s — how many, and of which entity
type, depends on how far extraction has run and on the LLM's own judgment calls, so the exact
counts will vary between runs. In one run over this corpus (Module 4's four-chunk extraction demo
plus Module 3's Wikidata enrichment for 3M and Apple), it produced 24 `EntityGroup`s: 6 `Company`,
5 `Product`, 6 `Regulation`, 4 `Risk`, 2 `FinancialMetric`, 1 `Location`, 0 `Person` (no
`Person`-type mention had been extracted yet in that run — the extraction demo batch never
touched an executive-heavy chunk).

The subsidiaries stayed correctly unmerged — the ~40 distinct `RecognisedEntity`/`Company` nodes
under "3M", each still `SUBSIDIARY_OF` its parent, none of them wrongly folded into the "3M"
group. The genuine duplicates — "3M"/"3M Company" across filings and against the curated
`Company` node, "Solventum"/"Solventum Corporation (Solventum)", "PFAS"/"PFAS compounds" — end up
sharing one `EntityGroup` each. One kind of case is a genuinely interesting judgment call worth
checking yourself in the output above: a Wikidata subsidiary item like `3M (Germany)` getting
grouped with the extracted legal-entity name `3M Deutschland GmbH` is plausibly correct, but
exactly the kind of cross-source call that's worth spot-checking rather than trusting blindly.

A downstream query for "everything about 3M" can now traverse
`(x)-[:SAME_AS]->(:EntityGroup {canonical_name: "3M"})<-[:SAME_AS]-(y)` and reach every mention
at once, instead of guessing which of several near-identical strings to search for.

Entities with only one mention stay ungrouped rather than being forced into a singleton group —
a group only gets written once there are at least two confirmed members. In an early-extraction
corpus like this one, most `Person`, `Product`, `Location`, `Regulation`, `Risk`, and
`FinancialMetric` mentions will fall into that bucket. Running `scripts/run_similarity.py` again
after a fuller `scripts/run_extraction.py` pass will pick those up as real duplicates appear.

That covers entities. The rest of this notebook turns to a related but distinct kind of
duplication: near-identical *chunk text*, not entities — and what a precomputed similarity graph
between chunks can do for retrieval quality.

## 10. Linking near-duplicate chunks by similarity

`similarity/linker.py::link_similar_chunks` searches the `chunk_embedding` vector index for each
`Chunk`'s nearest neighbors and writes `SIMILAR_TO {score}` edges for pairs scoring at or above
`threshold` (default 0.92 — chosen from the real score distribution: genuine near-repeats score
0.99-1.0, with a sharp drop to ~0.83-0.87 for merely topically-related content). Crucially, the
search is **not** scoped to one company or document — `retrieval/vector.py::semantic_search`
deliberately pre-filters by `company_id`/`year` for retrieval precision, but a linking pass wants
the opposite: an unscoped search is what surfaces duplicates *across* documents and companies,
not just within one filing.


In [ ]:
from financial_advisor.similarity.linker import link_similar_chunks

n_edges = link_similar_chunks()
print(f"{n_edges} SIMILAR_TO edge(s)")


In [ ]:
breakdown = neo4j_service.run_query(
    """
    MATCH (a:Chunk)-[r:SIMILAR_TO]->(b:Chunk)
    RETURN
      sum(CASE WHEN a.company_id <> b.company_id THEN 1 ELSE 0 END) AS cross_company,
      sum(CASE WHEN a.doc_id <> b.doc_id AND a.company_id = b.company_id THEN 1 ELSE 0 END) AS cross_doc_same_company,
      sum(CASE WHEN a.doc_id = b.doc_id THEN 1 ELSE 0 END) AS same_doc,
      count(*) AS total
    """
)[0]
print(breakdown)

example = neo4j_service.run_query(
    """
    MATCH (a:Chunk)-[r:SIMILAR_TO]->(b:Chunk)
    WHERE a.company_id <> b.company_id
    RETURN a.id AS a, b.id AS b, r.score AS score, left(a.text, 140) AS a_text
    ORDER BY r.score DESC
    LIMIT 1
    """
)[0]
print(f"\nHighest-scoring cross-company pair (score={example['score']:.4f}):")
print(f"  {example['a']}")
print(f"  {example['b']}")
print(f"  shared text: {example['a_text'].strip()}...")


**Illustrative numbers from one run over this corpus:** 756 edges — 169 within the same document
(a table or clause repeated within one filing), 519 across documents but the same company (a
paragraph reused between a company's 2024 and 2025 filing), and 68 genuinely cross-company — the
case a per-company-scoped search would have missed entirely. The exact counts depend on the
corpus, embedding model, and threshold, but the shape tends to hold: most near-duplicates are
within-document or within-company, with a smaller cross-company tail. The highest-scoring
cross-company pair above is standard SEC filing language, near-identical whether it's 3M or
Apple's 10-K.

## 11. Using SIMILAR_TO to avoid passing near-duplicate chunks

This is the motivating use case: retrieval routinely returns several near-duplicate chunks for
the same query (both filing years' business overview paragraphs, for instance) — wasted context
budget, and possibly a subtle bias in retrieval grading toward whatever's repeated. Recomputing
pairwise similarity among retrieved chunks at query time would work but repeats a cosine
comparison that's already sitting in the graph. `similarity/rerank.py::deduplicate_by_similarity`
instead does a cheap lookup: given a list of chunks (assumed ordered by relevance), fetch just
the `SIMILAR_TO` edges among *those* ids, and greedily drop a chunk once it's paired with an
already-kept, earlier — i.e. more relevant — chunk.


In [ ]:
from financial_advisor.retrieval.vector import semantic_search
from financial_advisor.similarity.rerank import deduplicate_by_similarity

chunks = semantic_search("3M business overview and company description", k=8, company_id="3M")
print("before dedup:")
for c in chunks:
    print(f"  {c['id']:35s} score={c['score']:.4f}")

deduped = deduplicate_by_similarity(chunks, threshold=0.92)
print("\nafter dedup:")
for c in deduped:
    print(f"  {c['id']:35s} score={c['score']:.4f}")
print(f"\n{len(chunks)} -> {len(deduped)} chunk(s)")


**Illustrative outcome, one run:** 8 chunks in, 5 out — the three dropped were each a
near-duplicate (score ≥ 0.92, via the edges just written) of a higher-ranked chunk already kept.
Nothing was recomputed; this was three graph lookups. The exact split between kept and dropped
chunks depends on what's actually retrieved for a given question and run.

### 11a. A real agent run, with and without dedup

`agent/graph.py::build_agent(dedup=True)` inserts `deduplicate_chunks_node` between `call_tools`
and `evaluate_retrieval` — the same before/after pattern Module 4 used for its new tools.
`dedup=False` (the default) is byte-for-byte the graph every earlier module already demonstrated.


In [ ]:
from financial_advisor.agent.graph import build_agent
from financial_advisor.agent.prompts import MODULE_3_STRATEGY_PROMPT
from financial_advisor.agent.state import initial_state
from financial_advisor.agent.tools import MODULE_3_TOOLS

agent_without = build_agent(MODULE_3_TOOLS, MODULE_3_STRATEGY_PROMPT, dedup=False)
agent_with = build_agent(MODULE_3_TOOLS, MODULE_3_STRATEGY_PROMPT, dedup=True)


def ask(agent, question: str) -> dict:
    result = agent.invoke(initial_state(question), {"recursion_limit": 50})
    tool_sequence = [entry["tool"] for entry in result["tool_call_log"]]
    print(f"[{result['retrieval_iterations']} retrieval round(s)] tools called: {tool_sequence}")
    print(f"[{len(result['retrieved_chunks'])} chunk(s) reached retrieval grading]")
    print(f"\nA: {result['answer']}")
    return result


QUESTION = "What does 3M's 10-K say about its principal businesses and where the company is headquartered?"


In [ ]:
print("===== WITHOUT dedup =====")
result_without = ask(agent_without, QUESTION)


In [ ]:
print("===== WITH dedup =====")
result_with = ask(agent_with, QUESTION)


**Illustrative trace, one run:** no year was specified, so a broad "business overview" question
left the *without* agent grinding through 3 retrieval rounds (`get_company_profile` →
`semantic_search` → `fulltext_search`) and carrying 15 chunks into retrieval grading before it
judged the growing knowledge sufficient. The *with* agent's very first `semantic_search` call
returned 5 chunks; `[dedup]` dropped 1 near-duplicate, leaving 4; grading judged that sufficient
immediately — 1 retrieval round, 4 chunks. Both agents produced correct, well-cited answers
(segments, products, headquarters address), so this run isn't a correctness difference — it's
that the *with* agent got there in a fraction of the rounds, on fewer chunks, because it wasn't
re-reading the same overview paragraph twice under two different chunk ids. The exact round and
chunk counts will vary with the question and the agent LLM's own choices from run to run, but the
pattern — fewer rounds, fewer chunks, same answer quality — is representative.

### 11b. Quantifying the savings: token usage

Round and chunk counts are a proxy; the number that actually maps to cost is tokens. LangChain's
`UsageMetadataCallbackHandler` captures real usage straight from the Azure OpenAI responses —
confirmed to propagate correctly through LangGraph's node calls, not just a direct `.invoke()` —
so this is measured, not estimated. Same two agents as 11a, same tools, same strategy prompt;
`dedup` is the only variable. Two questions, each run once per condition.


In [ ]:
from langchain_core.callbacks import UsageMetadataCallbackHandler

TOKEN_QUESTIONS = [
    QUESTION,
    "Summarize what 3M's 10-K says about its business overview, in a few sentences.",
]


def run_with_usage(question: str, dedup: bool) -> dict:
    agent = build_agent(MODULE_3_TOOLS, MODULE_3_STRATEGY_PROMPT, dedup=dedup)
    handler = UsageMetadataCallbackHandler()
    result = agent.invoke(initial_state(question), {"recursion_limit": 50, "callbacks": [handler]})
    usage = next(iter(handler.usage_metadata.values()), {})
    return {
        "rounds": result["retrieval_iterations"],
        "chunks": len(result["retrieved_chunks"]),
        "total_tokens": usage.get("total_tokens"),
    }


token_rows = []
for q in TOKEN_QUESTIONS:
    without = run_with_usage(q, dedup=False)
    with_ = run_with_usage(q, dedup=True)
    token_rows.append({"question": q, "without": without, "with": with_})


In [ ]:
print(f"{'question':50s} {'rounds':>10s} {'chunks':>10s} {'tokens':>16s} {'saved':>7s}")
for row in token_rows:
    w, d = row["without"], row["with"]
    saved_pct = (w["total_tokens"] - d["total_tokens"]) / w["total_tokens"] * 100
    q = row["question"]
    q_short = q if len(q) <= 50 else q[:47] + "..."
    rounds_str = f"{w['rounds']}->{d['rounds']}"
    chunks_str = f"{w['chunks']}->{d['chunks']}"
    tokens_str = f"{w['total_tokens']}->{d['total_tokens']}"
    print(f"{q_short:50s} {rounds_str:>10s} {chunks_str:>10s} {tokens_str:>16s} {saved_pct:>6.1f}%")

avg_saved_pct = sum(
    (r["without"]["total_tokens"] - r["with"]["total_tokens"]) / r["without"]["total_tokens"]
    for r in token_rows
) / len(token_rows) * 100
print(f"\naverage total-token reduction across {len(token_rows)} question(s): {avg_saved_pct:.1f}%")


**Illustrative numbers, one run.** The second question was the modest case: 2 near-duplicate
chunks dropped, tokens cut by roughly a fifth, both agents answered correctly in 1 round. The
first question was the dramatic case: the *without* agent's fulltext-search attempts for
"headquartered" kept missing, so it burned all the way to the retrieval-round cap (4 rounds, 10
accumulated chunks, 34,493 tokens) before answering; the *with* agent's very first
`semantic_search` call was sufficient after dropping 2 duplicates (1 round, 3 chunks, 8,821
tokens) — a 74% reduction that run. Both answers were still graded `accepted=True` on the first
attempt: the grading/answer LLMs were never *confused* by the duplicates in the without case,
they just paid — a lot more, this run — tokens circling back to find something already sitting in
the growing knowledge under a different chunk id.

That range — a modest reduction on an easy question, a dramatic one on a question where
redundancy tipped the without-agent into extra retrieval rounds — is the honest picture: token
savings from dedup aren't a fixed percentage, they depend on whether the duplicate happened to be
*why* retrieval grading asked for another round, and the exact figures will vary run to run with
the LLM's own choices. The claim that holds regardless is **cost efficiency at equal quality, not
a quality or noise improvement** — arguably the stronger, more defensible metric for a course
about explainable, production-grade retrieval, because it composes: a corpus-wide `SIMILAR_TO`
graph, computed once by `link_similar_chunks`, keeps paying this discount on every question asked
afterward, not just the two sampled here.

## 12. A research question: would Louvain clustering help? (Optional)

Louvain finds *communities* in a weighted graph — groups of nodes more densely connected to each
other than to the rest of the graph — by greedily optimizing a "modularity" score, not simply by
following edges until none are left (that's `gds.wcc`, weakly connected components). On a graph
with real substructure, the two diverge; on a sparse graph, they don't have room to.

The `SIMILAR_TO` graph just built (threshold 0.92) is exactly the sparse case: a few hundred edges
over roughly a thousand chunks, mostly isolated near-duplicate pairs. Worth checking empirically
before assuming Louvain adds anything here.

In [ ]:
DEDUP_GRAPH = "chunk-dedup-graph"
_ = neo4j_service.run_query("CALL gds.graph.drop($name, false)", {"name": DEDUP_GRAPH})
neo4j_service.run_query(
    "CALL gds.graph.project($name, 'Chunk', {SIMILAR_TO: {orientation: 'UNDIRECTED', properties: 'score'}})",
    {"name": DEDUP_GRAPH},
)

wcc = neo4j_service.run_query(
    "CALL gds.wcc.stream($name) YIELD componentId RETURN count(DISTINCT componentId) AS n",
    {"name": DEDUP_GRAPH},
)[0]["n"]
louvain = neo4j_service.run_query(
    "CALL gds.louvain.stream($name, {relationshipWeightProperty: 'score'}) YIELD communityId "
    "RETURN count(DISTINCT communityId) AS n",
    {"name": DEDUP_GRAPH},
)[0]["n"]
print(f"Dedup-threshold graph (0.92, 756 edges): WCC finds {wcc}, Louvain finds {louvain}")

_ = neo4j_service.run_query("CALL gds.graph.drop($name, false)", {"name": DEDUP_GRAPH})


**No difference, in the runs we've observed.** At the threshold tuned for near-duplicate
detection, Louvain tends to find about as many communities as plain connected components — there
isn't much modularity structure to optimize when the graph is this sparse, so running Louvain
here would just be a more expensive way to compute what `gds.wcc` already gives for free.
Conclusion: don't build a "cluster the dedup graph" feature — `rerank.py` doesn't need one
either, since it only ever looks at edges among a handful of already-retrieved chunks.

### 12a. A denser graph tells a different story

`similarity/clusters.py::detect_topic_clusters` builds a *separate*, deliberately looser graph
(threshold 0.75, uncapped neighbor count) via an ephemeral GDS Cypher projection — never written
to the stored graph — specifically to check whether Louvain earns its keep once there's actually
a dense, richly-connected structure to partition.


In [ ]:
LOOSE_GRAPH = "chunk-loose-demo"
_ = neo4j_service.run_query("CALL gds.graph.drop($name, false)", {"name": LOOSE_GRAPH})
neo4j_service.run_query(
    """
    CYPHER 25
    MATCH (c:Chunk)
    CALL (c) {
        MATCH (n:Chunk)
        SEARCH n IN (
            VECTOR INDEX chunk_embedding FOR c.embedding
            LIMIT 15
        ) SCORE AS score
        WHERE n <> c AND score >= 0.75 AND c.id < n.id
        RETURN n, score
    }
    WITH gds.graph.project(
        $name, c, n,
        {relationshipProperties: {score: score}},
        {undirectedRelationshipTypes: ['*']}
    ) AS g
    RETURN g.nodeCount AS nodeCount, g.relationshipCount AS relationshipCount
    """,
    {"name": LOOSE_GRAPH},
)[0]

wcc_loose = neo4j_service.run_query(
    "CALL gds.wcc.stream($name) YIELD componentId RETURN count(DISTINCT componentId) AS n",
    {"name": LOOSE_GRAPH},
)[0]["n"]
louvain_loose = neo4j_service.run_query(
    "CALL gds.louvain.stream($name, {relationshipWeightProperty: 'score'}) YIELD communityId "
    "RETURN count(DISTINCT communityId) AS n",
    {"name": LOOSE_GRAPH},
)[0]["n"]
print(f"Looser graph (0.75, uncapped): WCC finds {wcc_loose}, Louvain finds {louvain_loose}")

_ = neo4j_service.run_query("CALL gds.graph.drop($name, false)", {"name": LOOSE_GRAPH})


**The picture flips.** At this lower threshold the graph is dense enough that almost everything
is transitively reachable from everything else — `gds.wcc` collapses to essentially one giant
component, uninformative. `gds.louvain` on the *same* graph instead finds a double-digit number of
distinct communities, in the runs we've observed. This is the actual value modularity optimization
adds over connected components: on a densely connected graph, it still finds meaningful
sub-partitions instead of one undifferentiated blob.

In [ ]:
from financial_advisor.similarity.clusters import detect_topic_clusters

clusters = detect_topic_clusters()
print(f"{len(clusters)} topic cluster(s)\n")
for c in clusters[:8]:
    print(f"cluster {c['id']:4d} | {c['size']:3d} chunks | companies={c['companies']}")


In [ ]:
def sample_cluster_text(cluster, n=3):
    rows = neo4j_service.run_query(
        "MATCH (c:Chunk) WHERE c.id IN $ids RETURN c.company_id AS company, left(c.text, 130) AS snippet",
        {"ids": cluster["chunk_ids"][:n]},
    )
    return rows


for cluster in clusters[:2]:
    print(f"--- cluster {cluster['id']} ({cluster['size']} chunks, companies={cluster['companies']}) ---")
    for row in sample_cluster_text(cluster):
        print(f"  [{row['company']}] {row['snippet'].strip()}")
    print()


**What these communities can look like, one run:** the two largest clusters found were both
cross-company. The largest (133 chunks, both 3M and Apple) was dominated by boilerplate
certification and audit language — Sarbanes-Oxley Section 906 certifications, "Opinions on the
Financial Statements and Internal Control over Financial Reporting" — the standard-form legal
text every 10-K carries, near-identical regardless of which company filed it. The second-largest
(129 chunks, both companies) centered on company/business-description content — "Item 2.
Properties" (headquarters, owned/leased facilities) and "Company Background" narrative. The exact
cluster sizes and composition will vary run to run, but not perfectly pure topics (Louvain
optimizes modularity, not topic labels — a "properties and business description" cluster is a
coherent theme, not a single narrow subject) is a stable pattern, and clearly coherent,
cross-company clusters — the kind a per-document or per-company view would never surface — are
the point.

**Why this stays exploratory.** `detect_topic_clusters` deliberately doesn't write anything back
to the graph — no `Chunk.topic_cluster` property, no `TopicCluster` node. Persisting a threshold
and a schema decision for a feature nothing downstream reads yet would be exactly the kind of
premature structure this module already avoided for singleton entities. The connection worth
naming explicitly: community detection over a similarity/relationship graph, then per-community
summarization for retrieval, is the mechanism Microsoft's GraphRAG is built around (Leiden, a
Louvain refinement, over an extracted entity graph). This course is titled *Beyond* GraphRAG —
the point demonstrated here is that the same graph-native technique is available as a targeted
tool (topic discovery, corpus-wide boilerplate detection) without adopting it as the whole
retrieval architecture. If a future module wants topic-diverse retrieval or per-cluster
summaries, this is the starting point — not built further than the demonstration here.

## 13. Module 5, in full

Two independent pieces, both run for real against this corpus:

- **Entity resolution** (`resolver.py`) — `EntityGroup`s that correctly merge name variants of
  the same company/product/etc. while leaving `SUBSIDIARY_OF`-related entities apart; one run over
  this corpus produced 24 groups.
- **Chunk-similarity linking** (`linker.py`) — `SIMILAR_TO` edges, a meaningful fraction of them
  genuinely cross-company (756 edges, 68 cross-company in one run), feeding a real
  redundancy-aware re-ranking step (`rerank.py`, wired into the agent as `build_agent(dedup=True)`)
  and an exploratory Louvain topic-clustering pass (`clusters.py`).

Both `EntityGroup` and `SIMILAR_TO` are now real, queryable structure the rest of the course can
build on: Module 6's text2cypher can traverse `SAME_AS` for canonical-entity questions instead of
guessing which string variant to match, and any future retrieval work can call
`deduplicate_by_similarity` wherever redundant chunks would otherwise reach an LLM.